# Lesson 07 - รูปแบบการออกแบบการวางแผน

สมุดบันทึกนี้สาธิต **รูปแบบการออกแบบการวางแผน** สำหรับเอเย่นต์ AI โดยใช้ Microsoft Agent Framework
คุณจะได้เรียนรู้วิธีแบ่งคำขอเดินทางที่ซับซ้อนออกเป็นงานย่อยที่มีโครงสร้าง, มอบหมายให้กับเอเย่นต์ผู้เชี่ยวชาญ,
และดำเนินการตามแผนที่เกิดขึ้น — ทั้งหมดนี้โดยใช้ผลลัพธ์ที่มีโครงสร้างซึ่งสนับสนุนโดยโมเดล Pydantic.


## การตั้งค่า


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os, asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## การแบ่งงานออกเป็นส่วนย่อย

การแบ่งงานออกเป็นส่วนย่อยคือแกนหลักของรูปแบบการออกแบบการวางแผน แทนที่จะให้ตัวแทนเพียงคนเดียวจัดการคำร้องที่ซับซ้อนตั้งแต่ต้นจนจบ เราจะแบ่งปัญหาออกเป็น **งานย่อย** ที่เล็กลงและกำหนดไว้อย่างชัดเจน แต่ละงานย่อยจะถูกมอบหมายให้กับตัวแทนผู้เชี่ยวชาญเฉพาะด้าน (เช่น เที่ยวบิน โรงแรม กิจกรรม) ที่มีลำดับความสำคัญและลำดับความสัมพันธ์ของงานที่ชัดเจน

วิธีการนี้ให้ประโยชน์หลายประการ:
- **ความชัดเจน**: งานย่อยแต่ละงานมีหน้าที่รับผิดชอบเพียงอย่างเดียว
- **ความขนาน**: งานย่อยที่เป็นอิสระสามารถทำงานพร้อมกันได้
- **ความน่าเชื่อถือ**: ความล้มเหลวจะถูกจำกัดไว้เฉพาะงานย่อยแต่ละงาน
- **การติดตามงบประมาณ**: ค่าใช้จ่ายถูกประมาณสำหรับแต่ละงานย่อยและถูกรวมยอด


In [ ]:
class TravelSubTask(BaseModel):
    task_id: int
    description: str
    assigned_agent: str  # "flight_agent", "hotel_agent", "activity_agent"
    priority: str  # "high", "medium", "low"
    dependencies: list[int] = []


class TravelPlan(BaseModel):
    destination: str
    trip_duration_days: int
    subtasks: list[TravelSubTask]
    total_estimated_budget_usd: int
    notes: str

## การสร้างเอเย่นต์วางแผนด้วยผลลัพธ์ที่มีโครงสร้าง

เอเย่นต์วางแผนทำหน้าที่เป็น **ผู้ประสานงานแผนกต้อนรับ** เมื่อได้รับคำขอการเดินทางในระดับสูง จะสร้าง `TravelPlan` ที่มีโครงสร้าง — แยกคำขอออกเป็นงานย่อย กำหนดลำดับความสำคัญ และระบุความเชื่อมโยงกันเพื่อให้พนักงานผู้ดูแลหรืองานปฏิบัติการสามารถดำเนินการได้ต่อไป


In [ ]:
planning_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="""You are a travel planning agent. When given a travel request:
1. Break it into specific subtasks (flights, hotels, activities, logistics)
2. Assign each subtask to the appropriate specialist agent
3. Set priorities and identify dependencies between tasks
4. Estimate the total budget""",
)

result = await planning_agent.run(
    "Plan a 7-day trip to Paris for a couple interested in art, cuisine, and history. Budget around $5000.",
    )
if result:
    plan = result
    print(f"Destination: {plan.destination}")
    print(f"Duration: {plan.trip_duration_days} days")
    print(f"Budget: ${plan.total_estimated_budget_usd}")
    print(f"\nSubtasks:")
    for task in plan.subtasks:
        print(f"  [{task.priority}] {task.task_id}. {task.description} → {task.assigned_agent}")

## การดำเนินการตามแผนด้วยเครื่องมือผู้เชี่ยวชาญ

เมื่อเจ้าหน้าที่ต้อนรับได้จัดทำแผนอย่างเป็นระบบแล้ว **ตัวแทนคอนเซียร์จ** จะทำการดำเนินการตามแผนนั้น
เครื่องมือผู้เชี่ยวชาญแต่ละเครื่องมือจะจัดการกับหมวดหมู่ของงานย่อยหนึ่งประเภท (เที่ยวบิน โรงแรม กิจกรรม) คอนเซียร์จจะทำซ้ำผ่านงานย่อยในแผนตามลำดับความขึ้นต่อกันและส่งงานแต่ละอย่างไปยังเครื่องมือที่เหมาะสม


In [ ]:
@tool
def book_flight(
    destination: Annotated[str, "The destination city"],
    departure_date: Annotated[str, "Departure date (YYYY-MM-DD)"],
    return_date: Annotated[str, "Return date (YYYY-MM-DD)"],
) -> str:
    """Search and book flights for the trip."""
    return f"Flight booked to {destination}: {departure_date} → {return_date}, confirmation #FLT-{hash(destination) % 10000:04d}"


@tool
def reserve_hotel(
    city: Annotated[str, "The city for the hotel"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"],
) -> str:
    """Reserve a hotel room in the destination city."""
    return f"Hotel reserved in {city}: {check_in} to {check_out} for {guests} guests, confirmation #HTL-{hash(city) % 10000:04d}"


@tool
def book_activity(
    activity_name: Annotated[str, "Name of the activity or tour"],
    date: Annotated[str, "Date of the activity (YYYY-MM-DD)"],
    participants: Annotated[int, "Number of participants"],
) -> str:
    """Book a tour, museum visit, or other activity."""
    return f"Activity booked: {activity_name} on {date} for {participants} people, confirmation #ACT-{hash(activity_name) % 10000:04d}"


# Concierge agent that executes the plan using specialist tools
concierge_agent = await provider.create_agent(
    name="Concierge",
    instructions="""You are a travel concierge executing a structured travel plan.
Use the available tools to fulfil each subtask. Work through the subtasks in order,
respecting dependencies. Summarise the results when finished.""",
    tools=[book_flight, reserve_hotel, book_activity],
)

# Build a prompt from the plan produced above
if result.value:
    subtask_lines = "\n".join(
        f"- [{t.priority}] {t.task_id}. {t.description} (agent: {t.assigned_agent}, deps: {t.dependencies})"
        for t in plan.subtasks
    )
    execution_prompt = (
        f"Execute the following travel plan for {plan.destination} "
        f"({plan.trip_duration_days} days, ${plan.total_estimated_budget_usd} budget):\n"
        f"{subtask_lines}"
    )

    exec_response = await concierge_agent.run(execution_prompt)
    print(exec_response)

## สรุป

ในบทเรียนนี้คุณได้เรียนรู้เกี่ยวกับ **รูปแบบการออกแบบการวางแผน** สำหรับเอเจนต์ AI:

1. **การแยกงานออกเป็นส่วนย่อย** — เอเจนต์วางแผนที่โต๊ะหน้าแยกคำขอเดินทางที่ซับซ้อนออกเป็นงานย่อยที่มีโครงสร้างด้วยโมเดล Pydantic โดยมอบหมายแต่ละงานย่อยให้กับเอเจนต์ผู้เชี่ยวชาญพร้อมกับการกำหนดลำดับความสำคัญและความสัมพันธ์ระหว่างงาน
2. **ผลลัพธ์ที่มีโครงสร้าง** — โดยการส่ง `response_format` เอเจนต์จะส่งคืนวัตถุ `TravelPlan` ที่ผ่านการตรวจสอบแทนข้อความรูปแบบอิสระ ทำให้การประมวลผลต่อไปมีความน่าเชื่อถือ
3. **การดำเนินแผน** — เอเจนต์คอนเซียร์จวนวนผ่านงานย่อยโดยใช้เครื่องมือผู้เชี่ยวชาญ (`book_flight`, `reserve_hotel`, `book_activity`) เพื่อดำเนินการตามแผนและรายงานผลลัพธ์

รูปแบบนี้แยก *สิ่งที่ต้องทำ* (การวางแผน) ออกจาก *วิธีการทำ* (การดำเนินการ) ทำให้เอเจนต์มีความเป็นโมดูลมากขึ้น ทดสอบง่ายขึ้น และขยายระบบได้ง่ายขึ้น


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ข้อจำกัดความรับผิดชอบ**:
เอกสารนี้ได้รับการแปลโดยใช้บริการแปลภาษาอัตโนมัติ [Co-op Translator](https://github.com/Azure/co-op-translator) แม้เราจะพยายามให้ความถูกต้องสูงสุด แต่โปรดทราบว่าการแปลอัตโนมัติอาจมีข้อผิดพลาดหรือความไม่แม่นยำ เอกสารต้นฉบับในภาษาพื้นเมืองถือเป็นแหล่งข้อมูลที่เชื่อถือได้ สำหรับข้อมูลที่สำคัญ แนะนำให้ใช้บริการแปลโดยผู้เชี่ยวชาญมืออาชีพ เราไม่รับผิดชอบต่อความเข้าใจผิดหรือการตีความที่ผิดพลาดที่เกิดขึ้นจากการใช้การแปลนี้
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
